### Prepare environment

In [0]:
%run ../environment/prepare_environment

# Hyperparameter Tuning with Optuna

## Introduction
Hyperparameter tuning is crucial for getting the best performance out of your models. While Grid Search and Random Search are common, Bayesian optimization methods like **Optuna** are often more efficient.

In this notebook, we will:
1.  Load a classification dataset.
2.  Define an **Objective Function** for Optuna.
3.  Tune an **XGBoost Classifier** to maximize accuracy.
4.  Visualize the optimization history.

In [0]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

## 1. Defining the Objective Function

The objective function is the core of Optuna. It takes a `trial` object, suggests hyperparameters, trains the model, and returns the metric we want to optimize.

In [0]:
def objective(trial):
    # 1. Define hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        # Suggest categorical parameter
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        # Suggest float parameter (log scale is often useful for learning rates)
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True),
        # Suggest integer parameter
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    # 2. Train model with suggested params
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)

    # 3. Evaluate model
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)

    return accuracy

## 2. Running the Optimization

We create a `study` and tell Optuna to `maximize` the returned value (accuracy).

In [0]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Number of finished trials: ", len(study.trials))

## 3. Analyzing Results

Let's look at the best parameters found.

In [0]:
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

## 4. Visualization

Optuna provides built-in visualization plots to understand parameter importance and optimization history.

In [0]:
from optuna.visualization import plot_optimization_history, plot_param_importances

# Plot optimization history
plot_optimization_history(study).show()

# Plot parameter importance
plot_param_importances(study).show()